# Kapitel 1: Die Daten entdecken

## Kann eine Maschine Emotionen lesen?

Jeden Tag schreiben Millionen von Menschen Bewertungen auf Amazon.
In jedem dieser Texte steckt eine Emotion: Begeisterung, Enttäuschung, Wut oder Freude.

Unser Ziel: **Eine Maschine soll lernen, diese Emotionen automatisch zu erkennen.**

Doch bevor wir ein Modell trainieren können, müssen wir verstehen, womit wir arbeiten.
Dieses Notebook ist der erste Schritt — wir öffnen den Datensatz und schauen hinein.

## 1.1 SparkSession starten

Apache Spark ist unser Werkzeug für die Verarbeitung großer Datenmengen.
Anders als Pandas kann Spark Millionen von Zeilen effizient verarbeiten,
indem es die Arbeit auf mehrere Prozessorkerne verteilt.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviews – Daten laden") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark Version:", spark.version)
spark

## 1.2 Der erste Blick auf die Daten

Unser Datensatz stammt von Kaggle und enthält Amazon-Kundenbewertungen.
Die CSV-Datei hat **keinen Header** — wir vergeben die Spaltennamen manuell:

- **Score** — Sentiment-Label (1 = negativ, 2 = positiv)
- **Summary** — Kurztitel der Bewertung
- **Text** — Der vollständige Bewertungstext

Eine wichtige Erkenntnis: Die Scores sind **keine Sternebewertungen (1–5)**,
sondern bereits vorkategorisierte Sentiment-Labels. Das vereinfacht unsere spätere Arbeit.

In [ ]:
# CSV einlesen (OHNE Header)
df_raw = spark.read.csv(
    "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/data/test.csv",
    header=False,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

# Spalten umbenennen
df_raw = df_raw.toDF("Score", "Summary", "Text")

print(f"Datensatz erfolgreich geladen: {df_raw.count():,} Zeilen, {len(df_raw.columns)} Spalten")
print(f"Spalten: {df_raw.columns}")

### Was sehen wir?

400.000 Bewertungen — das ist viermal mehr als die Mindestanforderung von 100.000.
Schauen wir uns die Struktur genauer an.

In [ ]:
df_raw.printSchema()

In [ ]:
df_raw.show(5, truncate=80)

In [ ]:
df_raw.limit(5).toPandas()

## 1.3 Datenqualität: Gibt es Lücken?

Bevor wir weitermachen, müssen wir wissen: **Fehlen Daten?**
Fehlende Werte könnten unsere Analyse verfälschen.

In [ ]:
# Statistiken für die Score-Spalte
df_raw.select("Score").summary("count", "min", "max", "mean", "stddev").show()

In [ ]:
from pyspark.sql.functions import col, count, when, round as spark_round

total_rows = df_raw.count()

null_counts = df_raw.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df_raw.columns]
)

print(f"Gesamtanzahl der Zeilen: {total_rows:,}\n")
print("Fehlende Werte pro Spalte:")
print("-" * 40)

null_data = null_counts.collect()[0]
for col_name in df_raw.columns:
    null_val = null_data[col_name]
    pct = (null_val / total_rows) * 100
    print(f"{col_name:15s} → {null_val:>7,} ({pct:.2f}%)")

### Erkenntnis

Nur **4 fehlende Werte** in der Summary-Spalte bei 400.000 Zeilen — das ist verschwindend gering (0,001%).
Die Text-Spalte, die wir für die Analyse brauchen, ist komplett gefüllt. Ein guter Start.

## 1.4 Die zentrale Frage: Wie sind die Emotionen verteilt?

Die Score-Spalte bestimmt, ob eine Bewertung positiv oder negativ ist.
Ihre Verteilung ist entscheidend: Wenn eine Klasse stark überwiegt,
hat unser Modell es schwer, die Minderheit zu lernen.

In [ ]:
from pyspark.sql.functions import count as spark_count

score_dist = df_raw.groupBy("Score") \
    .agg(spark_count("*").alias("Anzahl")) \
    .orderBy("Score")

score_dist.show()

In [ ]:
import matplotlib.pyplot as plt

score_pd = score_dist.toPandas()

plt.figure(figsize=(8, 5))
bars = plt.bar(
    score_pd["Score"].astype(str),
    score_pd["Anzahl"],
    color="#5DCAA5",
    edgecolor="#0F6E56"
)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height):,}', ha='center', va='bottom', fontsize=10)

plt.xlabel("Bewertung (Score)")
plt.ylabel("Anzahl")
plt.title("Verteilung der Amazon-Bewertungen")
plt.tight_layout()
plt.show()

### Erkenntnis: Perfekte Balance

**200.000 negative und 200.000 positive Bewertungen — exakt 50/50.**

Das ist ideal für Machine Learning. In der Praxis sind Datensätze selten so
perfekt balanciert. Hier hat Kaggle die Daten bereits für uns vorbereitet.
Das bedeutet: Unser Modell wird nicht dazu neigen, eine Klasse zu bevorzugen.

## 1.5 Duplikate prüfen

In [ ]:
total = df_raw.count()
distinct = df_raw.dropDuplicates().count()
duplicates = total - distinct

print(f"Gesamtzeilen:      {total:>10,}")
print(f"Eindeutige Zeilen: {distinct:>10,}")
print(f"Duplikate:         {duplicates:>10,}")

## 1.6 Wie lang sind die Texte?

Für die spätere NLP-Verarbeitung ist die Textlänge wichtig.
Sehr kurze Texte enthalten wenig Information; sehr lange Texte
verlangsamen die Verarbeitung.

In [ ]:
from pyspark.sql.functions import length, avg, min as spark_min, max as spark_max

df_with_len = df_raw.withColumn("text_length", length(col("Text")))

df_with_len.select(
    spark_min("text_length").alias("Min"),
    spark_round(avg("text_length"), 0).alias("Durchschnitt"),
    spark_max("text_length").alias("Max")
).show()

In [ ]:
text_len_pd = df_with_len.select("text_length").toPandas()

plt.figure(figsize=(10, 5))
plt.hist(text_len_pd["text_length"].dropna(), bins=50, color="#85B7EB", edgecolor="#185FA5", range=(0, 3000))
plt.xlabel("Textlänge (Zeichen)")
plt.ylabel("Anzahl")
plt.title("Verteilung der Review-Textlängen (bis 3.000 Zeichen)")
plt.tight_layout()
plt.show()

## Kapitel 1 — Zusammenfassung

| Was wir gelernt haben | Detail |
|----------------------|--------|
| Datensatzgröße | 400.000 Bewertungen |
| Spalten | Score (Label), Summary, Text |
| Balance | Perfekt: 200K negativ, 200K positiv |
| Datenqualität | Nur 4 fehlende Werte, 0 Duplikate |
| Textlänge | Mehrheitlich zwischen 50 und 1.000 Zeichen |

**Die Daten sehen vielversprechend aus.** Wir haben einen großen, balancierten Datensatz
mit minimalen Qualitätsproblemen.

**Nächstes Kapitel:** Wir bereinigen die Daten — fehlende Werte entfernen, Text normalisieren
und alles für die maschinelle Verarbeitung vorbereiten.